# Biostrike: process a capture session on a free GPU

Turns a session recorded with the Biostrike capture server or iPhone app into the files the **Swing Optimizer** takes: a `.mot`, a `.trc` and the athlete's scaled `.osim` for every swing.

**Before you start**
1. Copy the whole session folder (the one with `Videos/Cam0`, `Videos/Cam1`, … and `sessionMetadata.yaml`) into your Google Drive, e.g. `My Drive/Biostrike/sessions/<session>`.
2. **Runtime → Change runtime type → T4 GPU**, then **Save**.
3. Fill in the session folder below and choose **Runtime → Run all**.

It takes about 10 minutes to set up, then 1–2 minutes per swing. Results land in `My Drive/Biostrike/results/<session>/<swing>/`; upload each swing folder to the Swing Optimizer.

If Colab disconnects, run all again: swings that already finished pose detection are skipped.

In [ ]:
#@title 1. Session to process
SESSION_FOLDER = "Biostrike/sessions/my_session"  #@param {type:"string"}
#@markdown Path inside **My Drive**. Leave the swings list empty to process every swing in the session.
SWINGS = ""  #@param {type:"string"}
#@markdown Code version (branch of the Biostrike fork):
BRANCH = "biostrike/local-capture"  #@param {type:"string"}

In [ ]:
#@title 2. Check the GPU and connect Google Drive
import os, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU:', gpu.stdout.strip())
    DEVICE = 'cuda:0'
else:
    print('No GPU on this runtime. Runtime → Change runtime type → T4 GPU, then run all again.')
    print('(Continuing on CPU would take 10–40 minutes per swing.)')
    DEVICE = 'cpu'

from google.colab import drive
drive.mount('/content/drive')
SESSION = os.path.join('/content/drive/MyDrive', SESSION_FOLDER.strip('/'))
RESULTS = os.path.join('/content/drive/MyDrive/Biostrike/results', os.path.basename(SESSION.rstrip('/')))
WEIGHTS = '/content/drive/MyDrive/Biostrike/weights'
assert os.path.isdir(os.path.join(SESSION, 'Videos')), f'No Videos folder in {SESSION}. Check SESSION_FOLDER.'
print('session:', SESSION)
print('results:', RESULTS)

In [ ]:
#@title 3. Get the Biostrike code
%cd /content
!rm -rf biostrike-capture
!git clone -q --depth 1 -b {BRANCH} https://github.com/jhayesf-arch/biostrike-capture.git
%cd /content/biostrike-capture
!git log -1 --format='code version: %h %s'

In [ ]:
#@title 4. Install the two environments (about 8 minutes)
#@markdown OpenCap's pose detector needs Python 3.8 with PyTorch 1.9; the rest of the pipeline needs
#@markdown Python 3.11 with OpenSim. They cannot share one environment, so each gets its own.
%cd /content
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!./bin/micromamba create -y -q -p /content/poseenv -c conda-forge python=3.8 pip > /dev/null
!./bin/micromamba create -y -q -p /content/pipeenv -c conda-forge python=3.11 pip > /dev/null
!/content/poseenv/bin/pip install -q -r /content/biostrike-capture/biostrike/requirements-pose-gpu.txt
!/content/pipeenv/bin/pip install -q -r /content/biostrike-capture/biostrike/requirements-pipeline.txt
!/content/poseenv/bin/python -c "import torch, mmpose; print('pose env: torch', torch.__version__, '| GPU visible:', torch.cuda.is_available())"
!/content/pipeenv/bin/python -c "import opensim; print('pipeline env: OpenSim', opensim.__version__)"

In [ ]:
#@title 5. Pose-model weights (downloaded once, kept in your Drive)
import hashlib, urllib.request
FILES = {
    'faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth':
        'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth',
    'hrnet_w48_coco_wholebody_384x288_dark-f5726563_20200918.pth':
        'https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w48_coco_wholebody_384x288_dark-f5726563_20200918.pth',
}
os.makedirs(WEIGHTS, exist_ok=True)
for name, url in FILES.items():
    path = os.path.join(WEIGHTS, name)
    if not os.path.exists(path):
        print('downloading', name)
        urllib.request.urlretrieve(url, path)
    # The id in each file name is the start of its SHA-256 hash.
    expected = name.rsplit('-', 1)[-1].split('.')[0]
    digest = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert digest.startswith(expected), f'{name} is corrupt: delete it from {WEIGHTS} and rerun'
    print('ok', name)

In [ ]:
#@title 6. Pose detection on the GPU
swings = SWINGS.replace(',', ' ').split()
# The neutral trial is detected too: it scales the athlete's model.
TRIAL_ARGS = ' '.join(['--trials'] + swings + ['neutral']) if swings else ''
!/content/poseenv/bin/python /content/biostrike-capture/biostrike/detect_session.py \
    --session "{SESSION}" --weights "{WEIGHTS}" --device {DEVICE} {TRIAL_ARGS}

In [ ]:
#@title 7. Calibration, triangulation, marker enhancer and OpenSim (CPU)
swing_args = ('--trials ' + ' '.join(swings)) if swings else ''
!cd /content/biostrike-capture && /content/pipeenv/bin/python biostrike/process_session.py \
    --session "{SESSION}" --results "{RESULTS}" --workdir /content/work {swing_args} \
    2>&1 | grep -v -i -E "warning|oneDNN|cuda|tensorflow"

In [ ]:
#@title 8. Results
for swing in sorted(os.listdir(RESULTS)) if os.path.isdir(RESULTS) else []:
    files = sorted(os.listdir(os.path.join(RESULTS, swing)))
    print(f'{swing:24s}', ', '.join(files))
print()
print('Upload each swing folder above to the Swing Optimizer (the .mot, .trc and .osim together).')
print('They are in Google Drive at:', RESULTS.replace('/content/drive/MyDrive', 'My Drive'))